# Bhojpuri SUMMARY — ZERO-SHOT — `llama31`

**Model:** `meta-llama/Llama-3.1-8B-Instruct` · **Test split:** `bho_test.csv` (218 rows) · **Shots:** 0

Your summary prompt is reproduced verbatim with only the exemplar block removed
(prompt sha1 of the string values = `66e49f74bc05f56d`). Affect-preservation clauses retained.

| setting | value | matches fine-tuned IndicBART |
|---|---|---|
| eval set | 218-row test split | yes |
| decoding | beams=4, len_pen=1.0, no_repeat=3 | yes |
| `max_new_tokens` | 200 | yes |
| precision | bfloat16 (no bitsandbytes) | — |
| post-processing | cleanup only, no length cap | yes (none) |
| empty output | scored 0, never dropped | yes |

**Before running:** Kernel → Restart, then Run All from the top. Cell 0 must run
before anything imports `torch`.


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [1]:
# ============================================================
# CELL 0 — GPU selection + environment. RUN FIRST, after Kernel -> Restart.
# ------------------------------------------------------------
# On a SHARED multi-GPU machine two things must happen BEFORE `import torch`:
#   1. choose ONE GPU (the one with most free memory) via CUDA_VISIBLE_DEVICES.
#      Setting it after torch is imported has no effect.
#   2. keep the model on that single GPU. device_map="auto" otherwise spreads
#      layers across every visible card — including ones other users have filled.
# ============================================================
import os, sys, shutil, subprocess

if "torch" in sys.modules:
    raise SystemExit("torch is already imported. Kernel -> Restart Kernel, then run this cell first.")

# ---- 1. find the GPU with the most free memory (nvidia-smi, no torch yet) ----
_rows = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,memory.free,memory.total,utilization.gpu",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip().split("\n")
_gpus = []
for r in _rows:
    i, free, tot, util = [x.strip() for x in r.split(",")]
    _gpus.append((int(i), int(free), int(tot), int(util)))
    print(f"  GPU {i}: {int(free)/1024:5.1f} GB free of {int(tot)/1024:.1f} GB | util {util}%")

GPU_ID, _free_mb, _tot_mb, _ = max(_gpus, key=lambda g: g[1])
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)           # BEFORE torch import
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"\n-> using physical GPU {GPU_ID} ({_free_mb/1024:.1f} GB free); "
      f"it appears as cuda:0 inside this kernel")

# ---- 2. environment (no compiler on some nodes -> bf16, no Triton) ----
for _k, _v in {"TORCHINDUCTOR_DISABLE":"1", "TORCHDYNAMO_DISABLE":"1",
               "TORCH_COMPILE_DISABLE":"1", "BITSANDBYTES_NOWELCOME":"1",
               "TOKENIZERS_PARALLELISM":"false", "HF_HUB_DISABLE_XET":"1",
               "HF_HUB_DOWNLOAD_TIMEOUT":"60"}.items():
    os.environ[_k] = _v
for _v in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    _cur = os.environ.get(_v, "")
    if "/lib64" not in _cur.split(":"):
        os.environ[_v] = f"/lib64:{_cur}" if _cur else "/lib64"
shutil.rmtree(os.path.expanduser("~/.triton/cache"), ignore_errors=True)

import torch
torch._dynamo.config.suppress_errors = True
assert torch.cuda.device_count() == 1, "more than one GPU still visible — restart the kernel"
_free, _total = torch.cuda.mem_get_info(0)
VRAM_GB = _free / (1024 ** 3)
print(f"torch {torch.__version__} | cuda:0 = {torch.cuda.get_device_name(0)} | "
      f"{VRAM_GB:.1f} GB free")


  GPU 0:  23.2 GB free of 24.0 GB | util 0%
  GPU 1:  15.7 GB free of 24.0 GB | util 0%

-> using physical GPU 0 (23.2 GB free); it appears as cuda:0 inside this kernel
torch 2.14.0+cu126 | cuda:0 = NVIDIA RTX A5000 | 23.0 GB free


In [3]:
# ============================================================
# CELL 1 — configuration (nothing to edit)
# ============================================================
MODEL_KEY  = "llama31"
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
ARCH       = "causal"         # causal | seq2seq
FMT        = "chat_sys"   # chat-template wrapper — the ONLY per-model difference
GATED      = True

LANG, TASK = "bho", "summary"
TEST_CSV, EXPECT_N = "bho_test.csv", 218

SHOTS            = 0
NUM_BEAMS        = 4
LENGTH_PENALTY   = 1.0
NO_REPEAT_NGRAM  = 3
DO_SAMPLE        = False
MAX_NEW_TOKENS   = 200          # same as fine-tuned IndicBART summary
MAX_INPUT_TOKENS = 1024
USE_4BIT         = False        # bf16: bitsandbytes needs a compiler this node lacks
BATCH_SIZE       = 1            # beams=4 x 1224-token context; raise only if memory allows
SEED             = 42
FRESH_START      = True

NEED_GB = 18
print(f"RUN {MODEL_KEY} | {LANG}/{TASK} | beams={NUM_BEAMS} max_new={MAX_NEW_TOKENS} "
      f"bf16 | shots={SHOTS}")
if VRAM_GB < NEED_GB:
    raise SystemExit(
        f"{MODEL_KEY} needs ~{NEED_GB} GB in bf16, but the freest GPU has only "
        f"{VRAM_GB:.1f} GB.\n"
        f"  1. check your OWN stale processes:  !nvidia-smi\n"
        f"     then Kernel -> Shut Down All Kernels (or kill that PID)\n"
        f"  2. or run a smaller model first: indicbart_pre (3 GB), sarvam1 (6 GB)")

RUN_TAG        = f"{MODEL_KEY}_{LANG}_{TASK}_zeroshot"
CHECKPOINT_CSV = f"ckpt_{RUN_TAG}.csv"
FINAL_CSV      = f"gen_{RUN_TAG}.csv"
PERITEM_CSV    = f"per_item_{RUN_TAG}.csv"
METRICS_JSON   = f"metrics_{RUN_TAG}.json"
MANIFEST_JSON  = f"manifest_{RUN_TAG}.json"


RUN llama31 | bho/summary | beams=4 max_new=200 bf16 | shots=0


In [5]:
# ============================================================
# CELL 2 — imports, seeding, GPU guard, HF auth (gated only)
# ============================================================
import re, gc, json, time, glob, random, hashlib, unicodedata
import numpy as np, pandas as pd

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "No GPU — a CPU fallback would take days."
print("device:", torch.cuda.get_device_name(0))

if GATED:
    from huggingface_hub import login, whoami, hf_hub_download
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("HF token (hidden): ").strip()
        os.environ["HF_TOKEN"] = tok
    login(tok, add_to_git_credential=False)
    print("logged in as:", whoami()["name"])
    try:
        hf_hub_download(MODEL_NAME, "config.json"); print("access OK")
    except Exception as e:
        raise SystemExit(f"No access to {MODEL_NAME}: {str(e)[:150]}\n"
                         f"Check https://huggingface.co/settings/gated-repos")


device: NVIDIA RTX A5000


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


logged in as: prakashdeepak132
access OK


In [6]:
# ============================================================
# CELL 3 — load the TEST SPLIT (self-healing)
# ============================================================
TEXT_COL, REF_COL, GEN_COL = "article_text", "summary", "generated"

cands = [TEST_CSV] + sorted(set(glob.glob("*test*.csv")))
path = next((p for p in cands if os.path.exists(p)), None)
if path is None:
    raise SystemExit(f"No test CSV in {os.getcwd()}. Expected {TEST_CSV}.")
if path != TEST_CSV:
    print(f"NOTE: using '{path}' (did not find {TEST_CSV})")

df = pd.read_csv(path, dtype=str, keep_default_na=False)
alias = {"article_text":["article_text","text","article","body","content"],
         "summary":["summary","gold_summary","reference_summary"],
         "headline":["headline","article_headline","title"]}
low = {c.strip().lower(): c for c in df.columns}
for want, alts in alias.items():
    if want not in df.columns:
        hit = next((low[a] for a in alts if a in low), None)
        if hit: df = df.rename(columns={hit: want}); print(f"  renamed {hit} -> {want}")
for c in (TEXT_COL, REF_COL):
    assert c in df.columns, f"missing '{c}'. Have: {list(df.columns)}"
    df[c] = df[c].astype(str).str.strip()
if "row_key" not in df.columns:
    df["row_key"] = df[TEXT_COL].map(lambda t: hashlib.sha1(t.encode()).hexdigest()[:16])
if not df["row_key"].is_unique:
    df["row_key"] = [f"{k}_{i}" for i, k in enumerate(df["row_key"])]

if len(df) != EXPECT_N:
    print(f"\n!!! {len(df)} rows, expected {EXPECT_N} (Table 6). Scores will NOT be "
          f"comparable to fine-tuned IndicBART. Continuing; true n is recorded.\n")
    EXPECT_N = len(df)
else:
    print(f"OK: {len(df)} rows matches Table 6")

df[GEN_COL] = ""
w = df[REF_COL].str.split().str.len()
print(f"gold summary words: mean={w.mean():.1f} median={w.median():.0f} "
      f"min={w.min()} max={w.max()}")


  renamed article_headline -> headline

!!! 217 rows, expected 218 (Table 6). Scores will NOT be comparable to fine-tuned IndicBART. Continuing; true n is recorded.

gold summary words: mean=79.5 median=80 min=40 max=140


In [7]:
# ============================================================
# CELL 4 — article cleaning
# ============================================================
_ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u200e\u200f\ufeff"), None)
def clean_article(t):
    s = "" if t is None else str(t)
    if s.strip().lower() in {"nan","none","null"}: return ""
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\t\n ")
    return re.sub(r"\s+", " ", s).strip()
df["_clean"] = df[TEXT_COL].map(clean_article)
print("empty articles:", int((df["_clean"] == "").sum()))


empty articles: 0


In [8]:
# ============================================================
# CELL 5 — PROMPT (your summary prompt, verbatim; exemplar block REMOVED)
# ============================================================
SYSTEM_PROMPT = 'रउआ एगो भोजपुरी समाचार सारांश लेखक बानी। रउआ हमेशा शुद्ध भोजपुरी में ही लिखेनी, हिंदी में कबो ना लिखेनी। रउआ लेख के मूल भाव आ भावना के जस के तस राखेनी।'

def _instruction_block():
    return 'नीचे दिहल भोजपुरी समाचार लेख के भोजपुरी भाषा में 80 से 100 शब्दन के सारांश लिखीं। सारांश खाली शुद्ध भोजपुरी में होखे के चाहीं — हिंदी में मत लिखीं। भोजपुरी के शब्द इस्तेमाल करीं जइसे: बा, बाड़ें, बानी, गइल, कइलस, रहल, होखे, चाहीं, रउआ, हमनी, ओकर, एकर।\n\nअउरी जरूरी बात:\n- सारांश 80 से 100 शब्दन के बीच में ही होखे। 100 शब्दन से जादे मत लिखीं।\n- लेख के भाव (खुशी, दुख, चिंता, गुस्सा, आस, निराशा, उत्साह) आ स्वर के जस के तस राखीं। लेख के भाव के हल्का भा साधारण मत करीं।\n- लेख में जवन नाम, जगह, संख्या आ तारीख बा, ओकरा के सारांश में शामिल करीं।\n- आपन ओर से कवनो नया जानकारी भा राय मत जोड़ीं। जवन लेख में बा, खाली ओकरे आधार पर लिखीं।\n- खाली एक बेर में एक साथ सारांश लिखीं। कवनो शीर्षक, ‘सारांश:’ जइसन शब्द, बुलेट भा नंबर मत जोड़ीं।'

def build_user_prompt(cleaned):
    """ZERO-SHOT. The original's 'एगो उदाहरण' exemplar block is deliberately absent."""
    return (_instruction_block()
            + f"\n\nलेख:\n{cleaned}\n\n"
            + "भोजपुरी सारांश (80-100 शब्द, खाली भोजपुरी में):")

def format_input(cleaned):
    u = build_user_prompt(cleaned)
    if FMT == "airavata":        # <|system|>/<|user|>/<|assistant|>, add_bos=False as in yours
        return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{u}\n<|assistant|>\n"
    if FMT == "plain":           # base LMs without a chat template
        return f"{SYSTEM_PROMPT}\n\n{u}"
    if FMT == "chat_sys":        # Llama-3.1
        return tokenizer.apply_chat_template(
            [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":u}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "chat_nosys":      # Gemma-2 has no system role
        return tokenizer.apply_chat_template(
            [{"role":"user","content":SYSTEM_PROMPT + "\n\n" + u}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "none":            # IndicBART: no instruction, language tag only
        return f"<2hi> {cleaned} </s>"
    raise ValueError(FMT)

PROMPT_SHA1 = hashlib.sha1((SYSTEM_PROMPT + _instruction_block()).encode()).hexdigest()[:16]
print("prompt sha1 =", PROMPT_SHA1, "  (expected 66e49f74bc05f56d)")
assert PROMPT_SHA1 == "66e49f74bc05f56d", "prompt text drifted from the verified version"
print("SHOTS = 0 -> no exemplar")


prompt sha1 = 66e49f74bc05f56d   (expected 66e49f74bc05f56d)
SHOTS = 0 -> no exemplar


In [9]:
# ============================================================
# CELL 6 — load model (bf16, eager attention, no bitsandbytes)
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, MBartForConditionalGeneration

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if ARCH == "seq2seq":
    model = MBartForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float32).cuda()
else:
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map={"": 0}, torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True, attn_implementation="eager")
model.eval()
print(f"loaded {MODEL_NAME} | {torch.cuda.memory_allocated()/1e9:.1f} GB")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded meta-llama/Llama-3.1-8B-Instruct | 16.1 GB


In [10]:
# ============================================================
# CELL 7 — post-processing (NO length cap) + generation
# ============================================================
_MARKERS = ("<|assistant|>","<|user|>","<|system|>","<|end|>","<|eot_id|>",
            "<|start_header_id|>","<|end_header_id|>","</s>","<s>","[/INST]",
            "<end_of_turn>","<start_of_turn>","<2hi>")

def postprocess(t):
    if not t: return ""
    s = str(t)
    for m in _MARKERS: s = s.replace(m, " ")
    s = re.sub(r"^\s*(भोजपुरी\s*)?(सारांश|summary)\s*[:：\-]\s*", "", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip().strip(' "\u201c\u201d')

NTOK = {}   # generated-token count per row, to measure truncation at the cap

@torch.no_grad()
def generate_batch(cleans):
    enc = tokenizer([format_input(c) for c in cleans], return_tensors="pt",
                    padding=True, truncation=True, max_length=MAX_INPUT_TOKENS,
                    add_special_tokens=(ARCH != "seq2seq")).to(model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS,
              length_penalty=LENGTH_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM,
              do_sample=DO_SAMPLE, early_stopping=True)
    if ARCH == "seq2seq":
        kw["decoder_start_token_id"] = tokenizer.convert_tokens_to_ids("<2hi>")
        out = model.generate(**enc, **kw)
        toks = out
    else:
        kw["pad_token_id"] = tokenizer.pad_token_id
        out = model.generate(**enc, **kw)
        toks = out[:, enc["input_ids"].shape[1]:]
    pad = tokenizer.pad_token_id
    lens = [int((row != pad).sum()) for row in toks]
    dec = tokenizer.batch_decode(toks, skip_special_tokens=True)
    return [postprocess(t) for t in dec], lens

t0 = time.time()
g, l = generate_batch(df["_clean"].head(2).tolist())
for i, (x, n) in enumerate(zip(g, l)):
    print(f"SMOKE {i}: [{n} tok, {len(x.split())} w] {x[:120]}")
est = (time.time() - t0) / 2 * len(df) / 60
print(f"est. ~{est:.0f} min for {len(df)} rows")


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


SMOKE 0: [66 tok, 26 w] ईराn के प्रधानमंत्री मोहम्मद रौहानी ने हाल के समय में कहा था की ईr के कोई भी हमले का जवाब देने की कोशिश नहीं की जाएगी.
SMOKE 1: [75 tok, 30 w] ुर्ति ने कहला कि हाल के दिनों में देखा गया है कि प्रदर्शनकारियों द्वारा अल्प संख्यकों पर हमला किया जा रहा है, जिससे कई ल
est. ~17 min for 217 rows


In [11]:
# ============================================================
# CELL 8 — generation loop (fresh start, checkpointed, OOM-safe)
# ============================================================
if os.path.exists(CHECKPOINT_CSV):
    if FRESH_START:
        os.rename(CHECKPOINT_CSV, f"{CHECKPOINT_CSV}.bak.{int(time.time())}")
        print("old checkpoint moved aside")
    else:
        ck = pd.read_csv(CHECKPOINT_CSV, dtype=str, keep_default_na=False)
        m = dict(zip(ck["row_key"], ck[GEN_COL])); df[GEN_COL] = df["row_key"].map(lambda k: m.get(k, ""))

todo = df.index[df[GEN_COL].str.strip() == ""].tolist()
print(f"generating {len(todo)}/{len(df)}")
t0 = time.time()
for s in range(0, len(todo), BATCH_SIZE):
    idx = todo[s:s+BATCH_SIZE]
    batch = df.loc[idx, "_clean"].tolist()
    try:
        outs, lens = generate_batch(batch)
    except Exception as e:
        print(f"  [batch {s}: {type(e).__name__}] retrying singly")
        torch.cuda.empty_cache(); outs, lens = [], []
        for a in batch:
            try:  o, n = generate_batch([a]); outs.append(o[0]); lens.append(n[0])
            except Exception: outs.append(""); lens.append(0)
    for i, o, n in zip(idx, outs, lens):
        df.at[i, GEN_COL] = o; NTOK[df.at[i, "row_key"]] = n
    df[["row_key", GEN_COL]].to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")
    if (s // BATCH_SIZE) % 10 == 0 or s + BATCH_SIZE >= len(todo):
        d = min(s + BATCH_SIZE, len(todo)); el = time.time() - t0
        print(f"  {d}/{len(todo)} | {el/60:.1f} min | ~{el/max(d,1)*(len(todo)-d)/60:.1f} min left")

print(f"done in {(time.time()-t0)/60:.1f} min | empty "
      f"{int((df[GEN_COL].str.strip()=='').sum())}/{len(df)}")
df.drop(columns=["_clean"]).to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
for i in df.index[:3]:
    print(f"\nGOLD: {df.at[i, REF_COL][:140]}\nGEN : {df.at[i, GEN_COL][:140]}")


old checkpoint moved aside
generating 217/217
  1/217 | 0.1 min | ~18.7 min left
  11/217 | 0.9 min | ~16.9 min left
  21/217 | 1.6 min | ~15.0 min left
  31/217 | 2.4 min | ~14.5 min left
  41/217 | 3.0 min | ~13.0 min left
  51/217 | 3.9 min | ~12.9 min left
  61/217 | 4.7 min | ~12.1 min left
  71/217 | 5.5 min | ~11.4 min left
  81/217 | 6.3 min | ~10.5 min left
  91/217 | 7.1 min | ~9.8 min left
  101/217 | 8.0 min | ~9.2 min left
  111/217 | 8.9 min | ~8.5 min left
  121/217 | 9.8 min | ~7.7 min left
  131/217 | 10.4 min | ~6.8 min left
  141/217 | 11.3 min | ~6.1 min left
  151/217 | 12.1 min | ~5.3 min left
  161/217 | 12.7 min | ~4.4 min left
  171/217 | 13.3 min | ~3.6 min left
  181/217 | 14.2 min | ~2.8 min left
  191/217 | 14.7 min | ~2.0 min left
  201/217 | 15.6 min | ~1.2 min left
  211/217 | 16.5 min | ~0.5 min left
  217/217 | 17.0 min | ~0.0 min left
done in 17.0 min | empty 1/217

GOLD: दुनिया में एगो अउरी बड़हन युद्ध शुरू होखे के संभावना बा। अमेरिकी राष्ट्रपति डोना

In [1]:
# ============================================================
# CELL 9 — SAVE GENERATIONS FIRST + STANDARDIZED EVALUATION
# ------------------------------------------------------------
# IMPORTANT:
# 1. Never regenerate anything here.
# 2. Preserve every generated summary exactly as saved by Cell 8.
# 3. Save a recovery copy BEFORE importing BERTScore/other metrics.
# 4. If one of the 218 rows is empty, keep it empty; do not delete/drop it.
# ============================================================

import os, json, shutil, time
import numpy as np
import pandas as pd

RUN_TAG = "llama31_bho_summary_zeroshot"
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
LANG = "bho"
TASK = "summary"
SHOTS = 0

REF_COL = "summary"
GEN_COL = "generated"
TEST_CSV = "bho_test.csv"
EXPECTED_N = 218

CHECKPOINT_CSV = f"ckpt_{RUN_TAG}.csv"
FINAL_CSV = f"gen_{RUN_TAG}.csv"
RECOVERY_CSV = f"gen_{RUN_TAG}_RECOVERY.csv"
PERITEM_CSV = f"per_item_{RUN_TAG}.csv"
METRICS_JSON = f"metrics_{RUN_TAG}.json"

# ------------------------------------------------------------
# 1. Recover the original test split
# ------------------------------------------------------------
ref = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)

if "row_key" not in ref.columns:
    import hashlib
    ref["row_key"] = ref["article_text"].astype(str).str.strip().map(
        lambda t: hashlib.sha1(t.encode("utf-8")).hexdigest()[:16]
    )

# ------------------------------------------------------------
# 2. Find the already-generated summaries.
#    Priority: in-memory df -> final CSV -> checkpoint CSV.
#    NOTHING IS REGENERATED.
# ------------------------------------------------------------
source = None

if "df" in globals() and GEN_COL in df.columns:
    source = "memory"
    ev = df.copy()
    print("Using generated summaries already in memory.")
else:
    candidates = [FINAL_CSV, CHECKPOINT_CSV]
    src = next((p for p in candidates if os.path.exists(p)), None)

    if src is None:
        raise FileNotFoundError(
            f"No saved generations found. Expected one of: {candidates}"
        )

    source = src
    gen = pd.read_csv(src, dtype=str, keep_default_na=False)

    if "row_key" not in gen.columns or GEN_COL not in gen.columns:
        raise ValueError(
            f"{src} must contain 'row_key' and '{GEN_COL}' columns."
        )

    # Keep the original test rows/order and attach the saved generations.
    ev = ref.copy()
    gen_map = dict(zip(gen["row_key"], gen[GEN_COL]))
    ev[GEN_COL] = ev["row_key"].map(lambda k: gen_map.get(k, ""))

    print(f"Recovered generations from: {src}")

# ------------------------------------------------------------
# 3. PRESERVE GENERATIONS EXACTLY AS THEY ARE
# ------------------------------------------------------------
ev[GEN_COL] = ev[GEN_COL].fillna("").astype(str)

# Save an untouched recovery copy BEFORE any evaluation.
ev.to_csv(RECOVERY_CSV, index=False, encoding="utf-8-sig")

# Also save the generation-only file in the standard location.
ev[["row_key", GEN_COL]].to_csv(
    FINAL_CSV, index=False, encoding="utf-8-sig"
)

print(f"\nSaved recovery copy: {RECOVERY_CSV}")
print(f"Saved generation file: {FINAL_CSV}")

# ------------------------------------------------------------
# 4. Report exactly what was generated
# ------------------------------------------------------------
N = len(ev)
empty_idx = [
    i for i, h in enumerate(ev[GEN_COL])
    if not str(h).strip()
]

print(f"\nGeneration status:")
print(f"  expected test rows : {EXPECTED_N}")
print(f"  rows in saved file : {N}")
print(f"  non-empty outputs  : {N - len(empty_idx)}")
print(f"  empty outputs      : {len(empty_idx)}")

if empty_idx:
    print(f"  empty row indices  : {empty_idx[:20]}")

if N != EXPECTED_N:
    print(
        f"\nWARNING: saved generation file has {N} rows, "
        f"while the test split has {EXPECTED_N} rows."
    )
    print("The saved generations will NOT be regenerated or discarded.")

# ------------------------------------------------------------
# 5. Only now run evaluation.
#    If BERTScore/torchvision has an environment error, the
#    recovery CSV above is already safely saved.
# ------------------------------------------------------------
refs = ev[REF_COL].fillna("").astype(str).str.strip().tolist()
hyps = ev[GEN_COL].fillna("").astype(str).str.strip().tolist()

# Do NOT assert N == EXPECTED_N here.
# A partially completed run must remain recoverable.
assert len(refs) == len(hyps), "Reference/generation length mismatch."

import sacrebleu
from rouge_score import rouge_scorer

class _WS:
    def tokenize(self, t):
        return str(t).split()

sc = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False,
    tokenizer=_WS()
)

def ci(v, n=1000):
    rng = np.random.default_rng(42)
    a = np.asarray(v, float)
    if len(a) == 0:
        return [float("nan"), float("nan")]
    b = [
        a[rng.integers(0, len(a), len(a))].mean()
        for _ in range(n)
    ]
    return [
        float(np.percentile(b, 2.5)),
        float(np.percentile(b, 97.5))
    ]

per = [sc.score(r, h) for r, h in zip(refs, hyps)]

r1 = [p["rouge1"].fmeasure for p in per]
r2 = [p["rouge2"].fmeasure for p in per]
rl = [p["rougeL"].fmeasure for p in per]

bl = [
    0.0 if not h else
    sacrebleu.sentence_bleu(
        h, [r], smooth_method="exp", tokenize="intl"
    ).score / 100
    for h, r in zip(hyps, refs)
]

cb = sacrebleu.corpus_bleu(hyps, [refs], tokenize="intl")

# ------------------------------------------------------------
# 6. BERTScore
#    If the current environment still has the ExportOptions/
#    torchvision problem, this import may fail. That will NOT
#    affect the saved generation CSVs above.
# ------------------------------------------------------------
bs = [0.0] * N

try:
    from bert_score import score as bert_score_fn

    ok = [i for i in range(N) if hyps[i] and refs[i]]

    if ok:
        _, _, F = bert_score_fn(
            [hyps[i] for i in ok],
            [refs[i] for i in ok],
            model_type="bert-base-multilingual-cased",
            batch_size=8,
            verbose=False
        )
        for j, i in enumerate(ok):
            bs[i] = float(F[j])

    print(f"BERTScore computed on {len(ok)}/{N} non-empty pairs.")

except Exception as e:
    print("\nWARNING: BERTScore could not be computed.")
    print(f"{type(e).__name__}: {e}")
    print("Generations are already safely saved.")
    print("Fix the BERTScore/PyTorch environment and rerun ONLY the evaluation section.")
    bs = [float("nan")] * N

# ------------------------------------------------------------
# 7. Aggregate metrics
# ------------------------------------------------------------
lens = [len(h.split()) for h in hyps]

M = {
    "run": RUN_TAG,
    "model": MODEL_NAME,
    "lang": LANG,
    "task": TASK,
    "shots": SHOTS,
    "n": N,
    "n_empty": len(empty_idx),
    "valid_rate": float(1 - len(empty_idx) / N) if N else 0.0,

    "bleu_sentence": float(np.mean(bl)) if N else 0.0,
    "bleu_sentence_ci": ci(bl),

    "rouge1": float(np.mean(r1)) if N else 0.0,
    "rouge1_ci": ci(r1),

    "rouge2": float(np.mean(r2)) if N else 0.0,
    "rouge2_ci": ci(r2),

    "rougeL": float(np.mean(rl)) if N else 0.0,
    "rougeL_ci": ci(rl),

    "bertscore_f1": (
        float(np.nanmean(bs))
        if not all(np.isnan(x) for x in bs)
        else None
    ),

    "len_mean": float(np.mean(lens)) if N else 0.0,
    "len_median": float(np.median(lens)) if N else 0.0,
    "len_std": float(np.std(lens)) if N else 0.0,

    "pct_80_100w": float(
        np.mean([80 <= l <= 100 for l in lens])
    ) if N else 0.0
}

# ------------------------------------------------------------
# 8. Save per-item evaluation without changing generations
# ------------------------------------------------------------
pd.DataFrame({
    "row_key": ev["row_key"],
    "ref": refs,
    "hyp": hyps,
    "rouge1": r1,
    "rouge2": r2,
    "rougeL": rl,
    "bleu": bl,
    "bertscore": bs,
    "len": lens
}).to_csv(
    PERITEM_CSV,
    index=False,
    encoding="utf-8-sig"
)

json.dump(
    M,
    open(METRICS_JSON, "w"),
    indent=2,
    ensure_ascii=False
)

print(f"\nSaved: {PERITEM_CSV}")
print(f"Saved: {METRICS_JSON}")


Recovered generations from: gen_llama31_bho_summary_zeroshot.csv

Saved recovery copy: gen_llama31_bho_summary_zeroshot_RECOVERY.csv
Saved generation file: gen_llama31_bho_summary_zeroshot.csv

Generation status:
  expected test rows : 218
  rows in saved file : 218
  non-empty outputs  : 214
  empty outputs      : 4
  empty row indices  : [23, 71, 189, 198]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore computed on 214/218 non-empty pairs.

Saved: per_item_llama31_bho_summary_zeroshot.csv
Saved: metrics_llama31_bho_summary_zeroshot.json


In [2]:
import os, json, glob, hashlib
import numpy as np, pandas as pd

# ---- 1. recover config if the kernel was restarted ----
_g = globals()
LANG      = _g.get("LANG", "bho")
TASK      = _g.get("TASK", "summary")
MODEL_KEY = _g.get("MODEL_KEY", "llama31")
SEED      = _g.get("SEED", 42)
MAX_NEW_TOKENS = _g.get("MAX_NEW_TOKENS", 200)
REF_COL   = _g.get("REF_COL", "summary")
GEN_COL   = _g.get("GEN_COL", "generated")
MODEL_NAME = _g.get("MODEL_NAME", "meta-llama/Llama-3.1-8B-Instruct")
SHOTS     = _g.get("SHOTS", 0)
RUN_TAG   = _g.get("RUN_TAG", "llama31_bho_summary_zeroshot")
TEST_CSV  = _g.get("TEST_CSV", f"{LANG}_test.csv")
PERITEM_CSV  = f"per_item_{RUN_TAG}.csv"
METRICS_JSON = f"metrics_{RUN_TAG}.json"
NTOK = _g.get("NTOK", {})

# ---- 2. get generations WITHOUT regenerating ----
if "df" in _g and GEN_COL in df.columns and (df[GEN_COL].astype(str).str.strip() != "").any():
    ev = df.copy()
    print("using generations already in memory")
else:
    ref = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)
    if "row_key" not in ref.columns:
        ref["row_key"] = ref["article_text"].astype(str).str.strip().map(
            lambda t: hashlib.sha1(t.encode()).hexdigest()[:16])
    src = next((p for p in (f"gen_{RUN_TAG}.csv", f"ckpt_{RUN_TAG}.csv") if os.path.exists(p)), None)
    if src is None:
        raise SystemExit(f"No saved generations for {RUN_TAG}. Looked for gen_/ckpt_ CSVs in {os.getcwd()}")
    g = pd.read_csv(src, dtype=str, keep_default_na=False)
    m = dict(zip(g["row_key"], g[GEN_COL]))
    ev = ref.copy()
    ev[GEN_COL] = ev["row_key"].map(lambda k: m.get(k, ""))
    print(f"loaded generations from {src}")

refs = ev[REF_COL].fillna("").astype(str).str.strip().tolist()
hyps = ev[GEN_COL].fillna("").astype(str).str.strip().tolist()
N = len(refs)
empty_idx = [i for i, h in enumerate(hyps) if not h]
print(f"n = {N} | empty generations = {len(empty_idx)}"
      + (f"  (rows {empty_idx[:10]})" if empty_idx else ""))

# ---- 3. ROUGE + BLEU (empty hyp -> 0 naturally) ----
import sacrebleu
from rouge_score import rouge_scorer
class _WS:
    def tokenize(self, t): return str(t).split()
sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"],
                              use_stemmer=False, tokenizer=_WS())
per = [sc.score(r, h) for r, h in zip(refs, hyps)]
r1 = [p["rouge1"].fmeasure for p in per]
r2 = [p["rouge2"].fmeasure for p in per]
rl = [p["rougeL"].fmeasure for p in per]
bl = [0.0 if not h else
      sacrebleu.sentence_bleu(h, [r], smooth_method="exp", tokenize="intl").score / 100
      for h, r in zip(hyps, refs)]
cb = sacrebleu.corpus_bleu(hyps, [refs], tokenize="intl")

# ---- 4. BERTScore on NON-EMPTY pairs only; empties get 0.0 ----
from bert_score import score as bert_score_fn
ok = [i for i in range(N) if hyps[i] and refs[i]]
bs = [0.0] * N
if ok:
    _, _, F = bert_score_fn([hyps[i] for i in ok], [refs[i] for i in ok],
                            lang="hi", model_type="bert-base-multilingual-cased",
                            batch_size=8, verbose=False)
    for j, i in enumerate(ok):
        bs[i] = float(F[j])
print(f"BERTScore computed on {len(ok)}/{N}; {N-len(ok)} empty scored 0.0")

# ---- 5. aggregate with bootstrap CIs ----
def ci(v, n=1000):
    rng = np.random.default_rng(SEED); a = np.asarray(v, float)
    b = [a[rng.integers(0, len(a), len(a))].mean() for _ in range(n)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]

lens = [len(h.split()) for h in hyps]
ntok = [NTOK.get(k, np.nan) for k in ev["row_key"]] if NTOK else [np.nan] * N
hit_cap = (float(np.nanmean([n >= MAX_NEW_TOKENS - 1 for n in ntok]))
           if NTOK else None)

M = {"run": RUN_TAG, "model": MODEL_NAME, "lang": LANG, "task": TASK, "shots": SHOTS,
     "n": N, "n_empty": len(empty_idx),
     "valid_rate": float(1 - len(empty_idx) / N),
     "bleu_sentence": float(np.mean(bl)), "bleu_sentence_ci": ci(bl),
     "rouge1": float(np.mean(r1)), "rouge1_ci": ci(r1),
     "rouge2": float(np.mean(r2)), "rouge2_ci": ci(r2),
     "rougeL": float(np.mean(rl)), "rougeL_ci": ci(rl),
     "bertscore_f1": float(np.mean(bs)), "bertscore_f1_ci": ci(bs),
     "bertscore_f1_nonempty": float(np.mean([bs[i] for i in ok])) if ok else 0.0,
     "brevity_penalty": float(cb.bp), "bleu_corpus": float(cb.score / 100),
     "len_mean": float(np.mean(lens)), "len_median": float(np.median(lens)),
     "len_std": float(np.std(lens)),
     "pct_80_100w": float(np.mean([80 <= l <= 100 for l in lens])),
     "pct_hit_token_cap": hit_cap}

print(f"\n{'='*64}\n{RUN_TAG}   n={N}   valid={M['valid_rate']:.1%}\n{'='*64}")
for k in ("bleu_sentence", "rouge1", "rouge2", "rougeL", "bertscore_f1"):
    lo, hi = M[k + "_ci"]
    print(f"  {k:<15} {M[k]:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  bertscore (non-empty only) {M['bertscore_f1_nonempty']:.4f}")
print(f"  brevity_penalty {M['brevity_penalty']:.4f}")
print(f"  length mean={M['len_mean']:.1f} med={M['len_median']:.0f} "
      f"in 80-100w={M['pct_80_100w']:.1%}")
if hit_cap is not None:
    print(f"  hit {MAX_NEW_TOKENS}-token cap: {hit_cap:.1%}")
else:
    print("  hit-token-cap: n/a (kernel restarted; token counts were in memory)")

pd.DataFrame({"row_key": ev["row_key"], "ref": refs, "hyp": hyps,
              "rouge1": r1, "rouge2": r2, "rougeL": rl, "bleu": bl,
              "bertscore": bs, "len": lens, "ntok": ntok}
             ).to_csv(PERITEM_CSV, index=False, encoding="utf-8-sig")
json.dump(M, open(METRICS_JSON, "w"), indent=2, ensure_ascii=False)
print(f"\nsaved {PERITEM_CSV}\nsaved {METRICS_JSON}")

if empty_idx:
    print(f"\nEmpty generations at test rows {empty_idx} — report valid_rate "
          f"({M['valid_rate']:.1%}) in the paper; these count as 0, not dropped.")


loaded generations from gen_llama31_bho_summary_zeroshot.csv
n = 218 | empty generations = 4  (rows [23, 71, 189, 198])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore computed on 214/218; 4 empty scored 0.0

llama31_bho_summary_zeroshot   n=218   valid=98.2%
  bleu_sentence   0.0047   95% CI [0.0037, 0.0057]
  rouge1          0.0821   95% CI [0.0731, 0.0906]
  rouge2          0.0028   95% CI [0.0019, 0.0039]
  rougeL          0.0635   95% CI [0.0571, 0.0692]
  bertscore_f1    0.6307   95% CI [0.6169, 0.6420]
  bertscore (non-empty only) 0.6424
  brevity_penalty 0.1266
  length mean=25.5 med=15 in 80-100w=1.4%
  hit-token-cap: n/a (kernel restarted; token counts were in memory)

saved per_item_llama31_bho_summary_zeroshot.csv
saved metrics_llama31_bho_summary_zeroshot.json

Empty generations at test rows [23, 71, 189, 198] — report valid_rate (98.2%) in the paper; these count as 0, not dropped.


In [ ]:
# ============================================================
# CELL 10 — protocol manifest (Section V-B) + failure-mode sample + cleanup
# ============================================================
json.dump({"run":RUN_TAG, "model_name":MODEL_NAME, "arch":ARCH, "chat_format":FMT,
           "language":LANG, "task":TASK, "eval_set":TEST_CSV, "n_eval":EXPECT_N,
           "split_seed":SEED, "reference_column":REF_COL, "shots":SHOTS, "exemplar":None,
           "prompt_sha1":PROMPT_SHA1, "affect_clause_in_prompt":True,
           "decoding":{"num_beams":NUM_BEAMS, "length_penalty":LENGTH_PENALTY,
                       "no_repeat_ngram_size":NO_REPEAT_NGRAM, "do_sample":DO_SAMPLE,
                       "max_new_tokens":MAX_NEW_TOKENS, "max_input_tokens":MAX_INPUT_TOKENS},
           "precision":"float32" if ARCH=="seq2seq" else "bfloat16",
           "attn_implementation":None if ARCH=="seq2seq" else "eager",
           "postprocessing":"marker + label cleanup only; NO length cap",
           "empty_output_policy":"scored as 0, never dropped",
           "metrics":{"rouge":"word-level whitespace, per-instance F1",
                      "bleu":"sentence-level, smooth=exp, tokenize=intl, /100",
                      "bertscore":"bert-base-multilingual-cased, lang=hi",
                      "ci":"1000-sample bootstrap, 95%"}},
          open(MANIFEST_JSON, "w"), indent=2, ensure_ascii=False)
print("saved", MANIFEST_JSON)

print("\n--- 15 outputs for failure-mode notes (Hindi? repetition? echo? truncated?) ---")
for i in df.index[:15]:
    h = df.at[i, GEN_COL]; k = df.at[i, "row_key"]
    print(f"[{len(h.split()):>3}w {NTOK.get(k,0):>3}tok] {h[:110] if h else '<EMPTY>'}")

del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print("\nGPU released.")
